# EduVidQA Eval — Gemini Unified 1-Call Judge

This notebook evaluates existing baseline and fine-tuned prediction files.

Design:
- Entailment remains local: `CrossEncoder("cross-encoder/nli-roberta-base")`
- Gemini is called **once per prediction** to produce:
  - FactQA Precision as `x/y`
  - FactQA Recall as `x/y`
  - Clarity 1–5
  - Critical Thinking 1–5
  - Pedagogical Techniques 1–5
- Custom rule metrics stay separate:
  - `format_pass_rate`, `no_think_rate`, `one_paragraph_rate`, markdown/code-fence rates
- Default Gemini thinking level: `HIGH`

Request count:
- Old official-style separate calls: `385 * 2 * 5 = 3850` Gemini calls
- This unified version: `385 * 2 * 1 = 770` Gemini calls

Claiming guidance:
- This is **EduVidQA-style unified evaluation**, not an exact reproduction of the repo's `metrics.py` call pattern.
- The metric definitions are preserved, but multiple LLM-based scores are batched into one structured judge call for API efficiency.

In [ ]:
# 0) Install eval dependencies
# Pin sentence-transformers==2.7.0 to avoid newer sentence_transformers -> torchcodec/FFmpeg import issues.
!pip uninstall -y torchcodec -q
!pip install -q "google-genai" "sentence-transformers==2.7.0" "pandas" "tqdm" "gdown"

In [ ]:
# 1) Config + mount Drive
from getpass import getpass
from pathlib import Path
import json, time, re, tarfile, os, math, statistics, random
from collections import defaultdict, Counter

import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")

# ===== Main control =====
FORCE_REJUDGE = False
RUN_BASELINE_JUDGE = True
RUN_FT_JUDGE = True
RUN_SUMMARY = True
RUN_RANDOM_REVIEW = True

# ===== Gemini judge config =====
JUDGE_PROVIDER = "gemini"
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
GEMINI_THINKING_LEVEL = "HIGH"  # MINIMAL, LOW, MEDIUM, HIGH
GEMINI_MAX_OUTPUT_TOKENS = 4096

GEMINI_RPM_LIMIT_PER_KEY = 15
GEMINI_RPD_LIMIT_PER_KEY = 500
GEMINI_RATE_LIMIT_COOLDOWN_SEC = 60
MAX_RETRIES_PER_SAMPLE = 6

GEMINI_API_KEY_1 = getpass("Gemini API key #1: ").strip()
GEMINI_API_KEY_2 = getpass("Gemini API key #2: ").strip()
GEMINI_API_KEY_3 = getpass("Gemini API key #3: ").strip()
GEMINI_API_KEYS = [GEMINI_API_KEY_1, GEMINI_API_KEY_2, GEMINI_API_KEY_3]
assert all(GEMINI_API_KEYS), "Nhập đủ 3 Gemini API keys."
assert len(set(GEMINI_API_KEYS)) == len(GEMINI_API_KEYS), "Ba API keys nên khác nhau để round-robin hiệu quả."

# ===== Drive paths from training notebook =====
WORKDIR = Path("/content/drive/MyDrive/eduvidqa_qwen35_vl_full_r16e1")
OUTPUT_DIR = WORKDIR / "outputs"

# Reuse predictions. These do not depend on judge provider/model/thinking.
BASELINE_PRED_PATH = OUTPUT_DIR / "baseline_full" / "predictions_full.jsonl"

FINAL_RUN_NAME = "final_fulltrain_h100_r16_e1_maxspeed"
FINAL_DIR = OUTPUT_DIR / "final_full_r16e1" / FINAL_RUN_NAME
FT_PRED_PATH = FINAL_DIR / "predictions_full.jsonl"

# Output path isolated by provider + model + thinking + unified protocol.
EVAL_RUN_TAG = f"{JUDGE_PROVIDER}_{GEMINI_MODEL}_thinking_{GEMINI_THINKING_LEVEL.lower()}_unified_1call"
EVAL_RUN_TAG = EVAL_RUN_TAG.replace("/", "_").replace(":", "_")
EVAL_ONLY_DIR = OUTPUT_DIR / "eval_official_eduvidqa_metrics" / EVAL_RUN_TAG
EVAL_ONLY_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUT_DIR = EVAL_ONLY_DIR

BASELINE_JUDGE_PATH = EVAL_ONLY_DIR / "baseline_unified_judge_full.jsonl"
FT_JUDGE_PATH = EVAL_ONLY_DIR / "fine_tuned_unified_judge_full.jsonl"

SUMMARY_OUT = EVAL_ONLY_DIR / "unified_eval_summary.json"
COMPARISON_CSV_OUT = EVAL_ONLY_DIR / "unified_eval_comparison.csv"
PAIRWISE_CSV_OUT = EVAL_ONLY_DIR / "unified_eval_pairwise_item_scores.csv"
HEADLINE_SUMMARY_CSV = EVAL_ONLY_DIR / "unified_eval_headline_summary.csv"
COMPLETION_STATUS_CSV = EVAL_ONLY_DIR / "unified_eval_completion_status.csv"
RANDOM_REVIEW_JSONL = EVAL_ONLY_DIR / "unified_random_review_examples.jsonl"
RANDOM_REVIEW_CSV = EVAL_ONLY_DIR / "unified_random_review_examples.csv"
RANDOM_REVIEW_MD = EVAL_ONLY_DIR / "unified_random_review_examples.md"

# ===== Dataset archive: references / question / GT answer =====
DATA_ARCHIVE_FILE_ID = "1uXvOVhwo8j944gRYBqpzxKZn0_EqjdzL"
DATA_ARCHIVE_FILENAME = "ft_context_vlm_clean.tar.gz"
DRIVE_DATA_CACHE = Path("/content/drive/MyDrive/eduvidqa")
DATASET_RUNTIME_ROOT = Path("/content/eduvidqa_eval_only_dataset")
DATASET_ROOT = DATASET_RUNTIME_ROOT / "ft_context_vlm_clean"

print("Judge provider:", JUDGE_PROVIDER)
print("Gemini model:", GEMINI_MODEL)
print("Gemini thinking:", GEMINI_THINKING_LEVEL)
print("Baseline pred:", BASELINE_PRED_PATH)
print("Baseline judge:", BASELINE_JUDGE_PATH)
print("FT pred:", FT_PRED_PATH)
print("FT judge:", FT_JUDGE_PATH)
print("Eval outputs:", EVAL_ONLY_DIR)

In [ ]:
# 2) Utility: robust JSON/JSONL IO

def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def read_jsonl_safe(path):
    path = Path(path)
    rows, bad = [], []
    if not path.exists():
        return rows

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                rows.append(json.loads(raw))
            except json.JSONDecodeError as exc:
                bad.append({"line_no": line_no, "error": str(exc), "prefix": raw[:300]})

    if bad:
        bad_path = path.with_suffix(path.suffix + ".bad_lines.json")
        write_json(bad_path, bad)
        print(f"[WARN] Skipped {len(bad)} broken JSONL lines in {path}; details -> {bad_path}")
    return rows

def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def overwrite_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_jsonl(path, rows):
    return overwrite_jsonl(path, rows)

def count_by_split(rows):
    return dict(Counter(r.get("split", "unknown") for r in rows))

def normalize_answer_text(pred):
    for key in ["prediction", "answer", "generated_answer", "response", "text", "output"]:
        val = pred.get(key)
        if isinstance(val, str) and val.strip():
            return val.strip()

    val = pred.get("messages")
    if isinstance(val, list):
        for m in reversed(val):
            if isinstance(m, dict) and m.get("role") in ["assistant", "model"] and m.get("content"):
                return str(m["content"]).strip()

    return ""

In [ ]:
# 3) Extract/load test references

DRIVE_DATA_CACHE.mkdir(parents=True, exist_ok=True)
archive_path = DRIVE_DATA_CACHE / DATA_ARCHIVE_FILENAME

if not archive_path.exists():
    print("Dataset archive not found on Drive; downloading with gdown...")
    !gdown {DATA_ARCHIVE_FILE_ID} -O {archive_path}

assert archive_path.exists(), f"Dataset archive not found: {archive_path}"

if not (DATASET_ROOT / "synthetic_test.jsonl").exists():
    print("Extracting dataset archive to local runtime...")
    DATASET_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(DATASET_RUNTIME_ROOT)
else:
    print("Dataset already extracted:", DATASET_ROOT)

synthetic_test = read_jsonl_safe(DATASET_ROOT / "synthetic_test.jsonl")
real_world_test = read_jsonl_safe(DATASET_ROOT / "real_world_test.jsonl")
full_test_records = synthetic_test + real_world_test

for r in synthetic_test:
    r.setdefault("split", "synthetic_test")
for r in real_world_test:
    r.setdefault("split", "real_world_test")

refs_by_id = {r["id"]: r for r in full_test_records}

print("synthetic_test:", len(synthetic_test))
print("real_world_test:", len(real_world_test))
print("full_test_records:", len(full_test_records))

In [ ]:
# 4) Load predictions and existing unified judge files

baseline_preds = read_jsonl_safe(BASELINE_PRED_PATH)
ft_preds = read_jsonl_safe(FT_PRED_PATH)

baseline_judged_existing = read_jsonl_safe(BASELINE_JUDGE_PATH)
ft_judged_existing = read_jsonl_safe(FT_JUDGE_PATH)

print("baseline_preds:", len(baseline_preds), count_by_split(baseline_preds))
print("ft_preds:", len(ft_preds), count_by_split(ft_preds))
print("baseline_judged_existing:", len(baseline_judged_existing), count_by_split(baseline_judged_existing))
print("ft_judged_existing:", len(ft_judged_existing), count_by_split(ft_judged_existing))

missing_refs_base = [p.get("id") for p in baseline_preds if p.get("id") not in refs_by_id]
missing_refs_ft = [p.get("id") for p in ft_preds if p.get("id") not in refs_by_id]

assert not missing_refs_base[:5], f"Missing references for baseline preds, e.g. {missing_refs_base[:5]}"
assert not missing_refs_ft[:5], f"Missing references for FT preds, e.g. {missing_refs_ft[:5]}"
assert len(baseline_preds) == len(ft_preds), "Baseline and FT prediction counts differ."

print("ready")

In [ ]:
# 5) Local custom rule metrics

def compute_rule_metrics(predictions):
    answers = [normalize_answer_text(p) for p in predictions]
    n = len(answers) or 1

    def has_markdown_bullets(a):
        return bool(re.search(r"(?m)^\s*([-*]|\d+[.)])\s+", a))

    def has_json_or_code_fence(a):
        s = a.strip()
        return "```" in s or s.startswith("{") or s.startswith("[") or s.lower().startswith("json")

    def one_paragraph(a):
        return "\n" not in a.strip()

    def format_pass(a):
        if not a.strip():
            return False
        if "<think>" in a.lower() or "</think>" in a.lower():
            return False
        if has_json_or_code_fence(a):
            return False
        if has_markdown_bullets(a):
            return False
        if not one_paragraph(a):
            return False
        return True

    word_counts = [len(a.split()) for a in answers]

    return {
        "sample_count": len(predictions),
        "format_pass_rate": sum(format_pass(a) for a in answers) / n,
        "no_think_rate": sum(("<think>" not in a.lower() and "</think>" not in a.lower()) for a in answers) / n,
        "one_paragraph_rate": sum(one_paragraph(a) for a in answers) / n,
        "markdown_bullet_rate": sum(has_markdown_bullets(a) for a in answers) / n,
        "json_or_code_fence_rate": sum(has_json_or_code_fence(a) for a in answers) / n,
        "empty_answer_rate": sum(not a.strip() for a in answers) / n,
        "answer_word_count_mean": sum(word_counts) / len(word_counts) if word_counts else None,
        "answer_word_count_p95": sorted(word_counts)[int(0.95 * (len(word_counts) - 1))] if word_counts else None,
        "latency_sec_mean": sum(float(p.get("latency_sec", 0) or 0) for p in predictions) / n,
        "output_tokens_mean": sum(float(p.get("output_tokens", 0) or 0) for p in predictions) / n,
    }

print("Baseline rule metrics:", compute_rule_metrics(baseline_preds))
print("FT rule metrics:", compute_rule_metrics(ft_preds))

In [ ]:
# 6) Gemini caller + local entailment model

from google import genai
from google.genai import types
from sentence_transformers import CrossEncoder
import torch

OFFICIAL_METRIC_FIELDS = [
    "entailment_score",
    "factqa_precision",
    "factqa_recall",
    "clarity",
    "critical_thinking",
    "pedagogical_techniques",
]

USE_CUDA_FOR_ENTAILMENT = False
ENTAILMENT_DEVICE = "cuda" if (USE_CUDA_FOR_ENTAILMENT and torch.cuda.is_available()) else "cpu"

print("Loading CrossEncoder entailment model on:", ENTAILMENT_DEVICE)
ent_model = CrossEncoder("cross-encoder/nli-roberta-base", device=ENTAILMENT_DEVICE)

UNIFIED_JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "factqa_precision_score": {
            "type": "string",
            "description": "Fraction '<supported>/<total>' for reference/gold claims supported by generated answer."
        },
        "factqa_recall_score": {
            "type": "string",
            "description": "Fraction '<supported>/<total>' for generated answer claims supported by reference/gold answer."
        },
        "clarity": {"type": "integer", "minimum": 1, "maximum": 5},
        "critical_thinking": {"type": "integer", "minimum": 1, "maximum": 5},
        "pedagogical_techniques": {"type": "integer", "minimum": 1, "maximum": 5},
        "rationale": {"type": "string", "description": "Short explanation under 80 words."}
    },
    "required": [
        "factqa_precision_score",
        "factqa_recall_score",
        "clarity",
        "critical_thinking",
        "pedagogical_techniques",
        "rationale"
    ]
}

def get_thinking_level(name: str):
    name = str(name or "HIGH").upper()
    valid = {"MINIMAL", "LOW", "MEDIUM", "HIGH", "THINKING_LEVEL_UNSPECIFIED"}
    if name not in valid:
        raise ValueError(f"Unsupported GEMINI_THINKING_LEVEL={name!r}; use {sorted(valid)}")
    return getattr(types.ThinkingLevel, name)

def is_rate_limit_error(exc):
    code = getattr(exc, "code", None)
    message = str(getattr(exc, "message", "") or exc).lower()
    return code == 429 or "rate limit" in message or "quota" in message or "resource exhausted" in message or "high demand" in message

def extract_json_obj(s):
    text = str(s or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\s*```$", "", text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass

    return None

class GeminiRoundRobinCaller:
    def __init__(self, api_keys, model_name, rpm_limit_per_key=15, rpd_limit_per_key=500, cooldown_sec=60):
        self.clients = [genai.Client(api_key=k) for k in api_keys]
        self.model_name = model_name
        self.rpm_limit = rpm_limit_per_key
        self.rpd_limit = rpd_limit_per_key
        self.cooldown_sec = cooldown_sec
        self.next_idx = 0
        self.calls = [0 for _ in api_keys]
        self.rate_limit_errors = [0 for _ in api_keys]
        self.rate_limited_until = [0.0 for _ in api_keys]
        self.last_call = [0.0 for _ in api_keys]
        self.usage = {
            "prompt_token_count": 0,
            "candidates_token_count": 0,
            "thoughts_token_count": 0,
            "total_token_count": 0,
        }

    def _take_slot(self):
        now = time.monotonic()
        for _ in range(len(self.clients)):
            idx = self.next_idx
            self.next_idx = (self.next_idx + 1) % len(self.clients)

            if self.calls[idx] >= self.rpd_limit:
                continue
            if self.rate_limited_until[idx] > now:
                continue

            min_gap = 60.0 / max(1, self.rpm_limit)
            wait = max(0.0, min_gap - (now - self.last_call[idx]))
            if wait:
                time.sleep(wait)

            self.calls[idx] += 1
            self.last_call[idx] = time.monotonic()
            return idx, self.clients[idx]

        wait_until = min(self.rate_limited_until) if self.rate_limited_until else time.monotonic() + self.cooldown_sec
        wait = max(5.0, wait_until - time.monotonic())
        print(f"All Gemini slots cooling down; sleeping {wait:.1f}s")
        time.sleep(wait)
        return self._take_slot()

    def _mark_rate_limited(self, idx, exc):
        self.rate_limit_errors[idx] += 1
        self.rate_limited_until[idx] = time.monotonic() + self.cooldown_sec
        print(f"Gemini key slot {idx} hit rate/quota/high-demand; switching key. Error: {str(exc)[:250]}")

    def _record_usage(self, response):
        usage = getattr(response, "usage_metadata", None)
        if usage is None:
            return

        for key in ["prompt_token_count", "candidates_token_count", "thoughts_token_count", "total_token_count"]:
            self.usage[key] += int(getattr(usage, key, 0) or 0)

    def call_unified_json(self, prompt, max_output_tokens=None, max_retries=None):
        max_output_tokens = max_output_tokens or GEMINI_MAX_OUTPUT_TOKENS
        max_retries = max_retries or MAX_RETRIES_PER_SAMPLE
        last_exc = None

        for attempt in range(1, max_retries + 1):
            idx, client = self._take_slot()
            try:
                config_kwargs = dict(
                    temperature=0,
                    max_output_tokens=max_output_tokens,
                    response_mime_type="application/json",
                    thinking_config=types.ThinkingConfig(
                        thinking_level=get_thinking_level(GEMINI_THINKING_LEVEL)
                    ),
                )

                # Some google-genai versions support response_schema; some do not.
                try:
                    config_kwargs["response_schema"] = UNIFIED_JUDGE_SCHEMA
                    config = types.GenerateContentConfig(**config_kwargs)
                except TypeError:
                    config_kwargs.pop("response_schema", None)
                    config = types.GenerateContentConfig(**config_kwargs)

                response = client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config=config,
                )
                self._record_usage(response)

                text = getattr(response, "text", None)
                if text is None:
                    text = str(response)

                obj = extract_json_obj(text)
                if isinstance(obj, dict):
                    return obj

                raise ValueError(f"Could not parse unified JSON. Prefix={text[:600]!r}")

            except Exception as exc:
                last_exc = exc
                if is_rate_limit_error(exc):
                    self._mark_rate_limited(idx, exc)
                    continue

                print(f"Unified Gemini JSON error attempt={attempt}: {str(exc)[:350]}")
                time.sleep(min(2 * attempt, 20))

        raise RuntimeError(f"Unified Gemini JSON call failed after {max_retries} attempts: {last_exc}")

gemini_caller = GeminiRoundRobinCaller(
    GEMINI_API_KEYS,
    GEMINI_MODEL,
    rpm_limit_per_key=GEMINI_RPM_LIMIT_PER_KEY,
    rpd_limit_per_key=GEMINI_RPD_LIMIT_PER_KEY,
    cooldown_sec=GEMINI_RATE_LIMIT_COOLDOWN_SEC,
)

def get_question_text(ref):
    for key in ["question", "query", "prompt", "user_question"]:
        if ref.get(key):
            return str(ref[key])
    return str(ref.get("text_input", ""))

def get_reference_answer(ref):
    for key in ["answer", "reference_answer", "gold_answer", "target"]:
        if ref.get(key):
            return str(ref[key])
    return ""

def get_candidate_answer(pred):
    return normalize_answer_text(pred)

print("Gemini unified 1-call metric engine ready.")
print("Expected Gemini calls for full baseline+FT:", len(baseline_preds) * 2 if "baseline_preds" in globals() else "load predictions first")

In [ ]:
# 7) Unified 1-call EduVidQA-style metric functions

def parse_fraction_score(score_str):
    s = str(score_str or "").strip()
    m = re.search(r"(\d+)\s*/\s*(\d+)", s)
    if not m:
        return None
    num, den = int(m.group(1)), int(m.group(2))
    if den <= 0:
        return None
    return num / den

def clamp_int(x, low, high):
    try:
        x = int(x)
    except Exception:
        return low
    return max(low, min(high, x))

def official_entailment(question, answer, generated):
    sentence_pair = (generated, answer)
    scores = ent_model.predict(sentence_pair)
    probs = torch.nn.functional.softmax(torch.tensor(scores), dim=-1).tolist()
    return float(probs[1])

def build_unified_official_prompt(question, reference_answer, generated_answer):
    return f"""
You are a strict evaluator for educational video question answering.

Evaluate a generated answer against a reference/gold answer using EduVidQA-style metrics.

Think carefully. Return only valid JSON matching this exact schema:
{json.dumps(UNIFIED_JUDGE_SCHEMA, ensure_ascii=False)}

Question:
{question}

Reference / Gold Answer:
{reference_answer}

Generated Answer:
{generated_answer}

Metric 1 - FactQA Precision:
Evaluate atomic claims made by the Reference / Gold Answer against the Generated Answer.
Steps:
1. List atomic claims made by the Reference / Gold Answer.
2. Count how many are supported by the Generated Answer.
3. Return factqa_precision_score as "<supported>/<total>".

Metric 2 - FactQA Recall:
Evaluate atomic claims made by the Generated Answer against the Reference / Gold Answer.
Steps:
1. List atomic claims made by the Generated Answer.
2. Count how many are supported by the Reference / Gold Answer.
3. Return factqa_recall_score as "<supported>/<total>".

Metric 3 - Clarity, score 1-5:
1 = severe unclear writing, multiple unexplained jargon terms, incoherent transitions.
2 = at least one unexplained jargon term plus a logical jump or ambiguous phrasing.
3 = mostly clear with one or two minor issues.
4 = all terms explained, clear flow, at most one small unclear phrase.
5 = no unexplained jargon, consistent logical flow, no ambiguity.

Metric 4 - Encouraging Critical Thinking, score 1-5:
1 = purely factual; no questions, alternatives, or exploration prompts.
2 = one suggestive or reflective phrase, but no actual open-ended question.
3 = one open-ended question or one alternative method/viewpoint.
4 = at least two open-ended prompts or multiple viewpoints briefly compared.
5 = at least two open-ended questions plus explicit invitation to explore further.

Metric 5 - Using Pedagogical Techniques, score 1-5:
1 = pure explanation without example, analogy, or breakdown.
2 = one brief example or partial step list, weak or incomplete.
3 = one complete example or one full step list, but not both.
4 = at least two teaching techniques, such as example plus step list.
5 = at least three techniques, such as example, analogy, step list, or visual cue, all clear and complete.

Important:
- Do not add markdown.
- Do not include hidden reasoning.
- The rationale field must be one short sentence under 80 words.
- Return only the JSON object.
""".strip()

def official_get_all_scores(ref, pred):
    question = get_question_text(ref)
    answer = get_reference_answer(ref)
    generated = get_candidate_answer(pred)

    ent = official_entailment(question, answer, generated)
    prompt = build_unified_official_prompt(question, answer, generated)
    obj = gemini_caller.call_unified_json(prompt, max_output_tokens=GEMINI_MAX_OUTPUT_TOKENS)

    fqa_p = parse_fraction_score(obj.get("factqa_precision_score"))
    fqa_r = parse_fraction_score(obj.get("factqa_recall_score"))

    if fqa_p is None:
        fqa_p = 0.0
    if fqa_r is None:
        fqa_r = 0.0

    clarity = clamp_int(obj.get("clarity"), 1, 5)
    critical = clamp_int(obj.get("critical_thinking"), 1, 5)
    pedagogy = clamp_int(obj.get("pedagogical_techniques"), 1, 5)

    return {
        "entailment_score": ent,
        "factqa_precision": float(fqa_p),
        "factqa_recall": float(fqa_r),
        "clarity": clarity,
        "critical_thinking": critical,
        "pedagogical_techniques": pedagogy,
        "factqa_precision_score_raw": obj.get("factqa_precision_score"),
        "factqa_recall_score_raw": obj.get("factqa_recall_score"),
        "judge_rationale": obj.get("rationale", ""),
        "hallucination_proxy_1_minus_factqa_precision": 1.0 - float(fqa_p),
    }

print("Unified 1-call metric functions ready.")

In [ ]:
# 8) Run unified 1-call judge with resume

def judge_predictions(predictions, out_path, candidate_label, force=False):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if force and out_path.exists():
        backup = out_path.with_suffix(out_path.suffix + f".backup_{int(time.time())}")
        out_path.rename(backup)
        print("Backed up old judge file:", backup)

    rows = [] if force else read_jsonl_safe(out_path)
    done = {r.get("id") for r in rows if r.get("judge_status", "ok") == "ok"}

    print(f"{candidate_label}: existing judged OK={len(done)} / predictions={len(predictions)}")
    print("Output:", out_path)

    for pred in tqdm(predictions, desc=f"Unified judge {candidate_label}", unit="sample"):
        pred_id = pred.get("id")
        if pred_id in done:
            continue

        ref = refs_by_id[pred_id]
        try:
            scores = official_get_all_scores(ref, pred)
            row = {
                "id": pred_id,
                "split": ref.get("split"),
                "video_id": ref.get("video_id"),
                "candidate_label": candidate_label,
                "judge_status": "ok",
                "metric_protocol": "eduvidqa_official_style_unified_1call_gemini",
                "judge_provider": JUDGE_PROVIDER,
                "judge_model": GEMINI_MODEL,
                "gemini_thinking_level": GEMINI_THINKING_LEVEL,
                **scores,
            }
        except Exception as exc:
            row = {
                "id": pred_id,
                "split": ref.get("split"),
                "video_id": ref.get("video_id"),
                "candidate_label": candidate_label,
                "judge_status": "failed",
                "metric_protocol": "eduvidqa_official_style_unified_1call_gemini",
                "judge_error": str(exc)[:1000],
            }
            print("[WARN] unified judge failed for", pred_id, "error:", exc)

        append_jsonl(out_path, row)
        rows.append(row)

        if row.get("judge_status") == "ok":
            done.add(pred_id)

    print(f"saved {len(rows)} rows -> {out_path}")
    return read_jsonl_safe(out_path)

baseline_judged = baseline_judged_existing
ft_judged = ft_judged_existing

if RUN_BASELINE_JUDGE:
    baseline_judged = judge_predictions(baseline_preds, BASELINE_JUDGE_PATH, "baseline", force=FORCE_REJUDGE)

if RUN_FT_JUDGE:
    ft_judged = judge_predictions(ft_preds, FT_JUDGE_PATH, "fine_tuned", force=FORCE_REJUDGE)

print("Gemini calls per key:", gemini_caller.calls)
print("Gemini rate-limit/high-demand errors per key:", gemini_caller.rate_limit_errors)
print("Gemini token usage so far:", json.dumps(gemini_caller.usage, ensure_ascii=False, indent=2))

In [ ]:
# 9) Aggregate unified official-style metrics + custom rule metrics

OFFICIAL_NUMERIC_FIELDS = [
    "entailment_score",
    "factqa_precision",
    "factqa_recall",
    "clarity",
    "critical_thinking",
    "pedagogical_techniques",
    "hallucination_proxy_1_minus_factqa_precision",
]

def ok_judged(rows):
    return [r for r in (rows or []) if isinstance(r, dict) and r.get("judge_status", "ok") == "ok"]

def _safe_float(x):
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None

def _aggregate_flat(rows):
    rows = ok_judged(rows)
    out = {"sample_count": len(rows)}
    for field in OFFICIAL_NUMERIC_FIELDS:
        vals = [_safe_float(r.get(field)) for r in rows]
        vals = [v for v in vals if v is not None]
        out[field + "_mean"] = sum(vals) / len(vals) if vals else None
        out[field + "_count"] = len(vals)
    return out

def aggregate_judge_metrics(rows, include_by_split=True):
    rows = ok_judged(rows)
    out = _aggregate_flat(rows)
    if include_by_split:
        out["by_split"] = {}
        for split in sorted(set(r.get("split", "unknown") for r in rows)):
            sub = [r for r in rows if r.get("split", "unknown") == split]
            out["by_split"][split] = _aggregate_flat(sub)
    return out

def official_composite_score(row):
    if row.get("judge_status", "ok") != "ok":
        return None
    ent = float(row.get("entailment_score", 0) or 0)
    fqa_p = float(row.get("factqa_precision", 0) or 0)
    fqa_r = float(row.get("factqa_recall", 0) or 0)
    clarity = float(row.get("clarity", 0) or 0) / 5
    critical = float(row.get("critical_thinking", 0) or 0) / 5
    pedagogy = float(row.get("pedagogical_techniques", 0) or 0) / 5
    return 0.25 * ent + 0.25 * fqa_p + 0.15 * fqa_r + 0.15 * clarity + 0.10 * pedagogy + 0.10 * critical

def paired_win_rate(base_rows, ft_rows, eps=0.015):
    base_map = {r["id"]: r for r in ok_judged(base_rows) if r.get("id") is not None}
    ft_map = {r["id"]: r for r in ok_judged(ft_rows) if r.get("id") is not None}
    ids = sorted(set(base_map) & set(ft_map))

    records = []
    win = tie = lose = 0
    by_split = defaultdict(lambda: {"compared": 0, "win": 0, "tie": 0, "lose": 0})

    for id_ in ids:
        b, f = base_map[id_], ft_map[id_]
        b_score = official_composite_score(b)
        f_score = official_composite_score(f)
        if b_score is None or f_score is None:
            continue

        delta = f_score - b_score
        if delta > eps:
            verdict = "win"; win += 1
        elif delta < -eps:
            verdict = "lose"; lose += 1
        else:
            verdict = "tie"; tie += 1

        split = f.get("split", b.get("split", "unknown"))
        by_split[split]["compared"] += 1
        by_split[split][verdict] += 1

        records.append({
            "id": id_,
            "split": split,
            "baseline_official_composite": b_score,
            "fine_tuned_official_composite": f_score,
            "delta": delta,
            "verdict": verdict,
            "baseline_entailment": b.get("entailment_score"),
            "fine_tuned_entailment": f.get("entailment_score"),
            "baseline_factqa_precision": b.get("factqa_precision"),
            "fine_tuned_factqa_precision": f.get("factqa_precision"),
            "baseline_factqa_recall": b.get("factqa_recall"),
            "fine_tuned_factqa_recall": f.get("factqa_recall"),
            "baseline_clarity": b.get("clarity"),
            "fine_tuned_clarity": f.get("clarity"),
            "baseline_critical_thinking": b.get("critical_thinking"),
            "fine_tuned_critical_thinking": f.get("critical_thinking"),
            "baseline_pedagogical_techniques": b.get("pedagogical_techniques"),
            "fine_tuned_pedagogical_techniques": f.get("pedagogical_techniques"),
        })

    overall = {
        "compared": len(records),
        "win": win,
        "tie": tie,
        "lose": lose,
        "win_rate": win / len(records) if records else None,
        "tie_rate": tie / len(records) if records else None,
        "lose_rate": lose / len(records) if records else None,
    }

    by_split_out = {}
    for split, d in by_split.items():
        c = d["compared"]
        by_split_out[split] = {
            **d,
            "win_rate": d["win"] / c if c else None,
            "tie_rate": d["tie"] / c if c else None,
            "lose_rate": d["lose"] / c if c else None,
        }

    return overall, by_split_out, pd.DataFrame(records)

def summary_to_dataframe(summary):
    rows = []
    for group_name in ["rule_metrics", "official_judge_metrics"]:
        base = summary[group_name]["baseline"]
        ft = summary[group_name]["fine_tuned"]
        for k in sorted(set(base) & set(ft)):
            if k == "by_split":
                continue
            if isinstance(base[k], (int, float)) and isinstance(ft[k], (int, float)):
                rows.append({
                    "metric": group_name.replace("_metrics", "") + "." + k,
                    "baseline": base[k],
                    "fine_tuned": ft[k],
                    "delta": ft[k] - base[k],
                })
    return pd.DataFrame(rows)

if RUN_SUMMARY:
    baseline_judge_metrics = aggregate_judge_metrics(baseline_judged)
    ft_judge_metrics = aggregate_judge_metrics(ft_judged)
    overall_pair, by_split_pair, pair_df = paired_win_rate(baseline_judged, ft_judged)

    summary = {
        "metric_protocol": "eduvidqa_official_style_unified_1call_plus_custom_format_rules_gemini",
        "config": {
            "judge_provider": JUDGE_PROVIDER,
            "judge_model": GEMINI_MODEL,
            "gemini_thinking_level": GEMINI_THINKING_LEVEL,
            "gemini_keys": len(GEMINI_API_KEYS),
            "rpm_per_key": GEMINI_RPM_LIMIT_PER_KEY,
            "rpd_per_key": GEMINI_RPD_LIMIT_PER_KEY,
            "rate_limit_errors_per_key": gemini_caller.rate_limit_errors,
            "gemini_usage": gemini_caller.usage,
            "entailment_model": "cross-encoder/nli-roberta-base",
            "entailment_device": ENTAILMENT_DEVICE,
            "note": "Unified judge combines FactQA P/R and qualitative rubrics into one Gemini call per prediction for API efficiency.",
        },
        "paths": {
            "baseline_predictions_full": str(BASELINE_PRED_PATH),
            "baseline_judge_full": str(BASELINE_JUDGE_PATH),
            "fine_tuned_predictions_full": str(FT_PRED_PATH),
            "fine_tuned_judge_full": str(FT_JUDGE_PATH),
            "summary_out": str(SUMMARY_OUT),
            "comparison_csv_out": str(COMPARISON_CSV_OUT),
            "pairwise_csv_out": str(PAIRWISE_CSV_OUT),
        },
        "rule_metrics": {
            "baseline": compute_rule_metrics(baseline_preds),
            "fine_tuned": compute_rule_metrics(ft_preds),
        },
        "official_judge_metrics": {
            "baseline": baseline_judge_metrics,
            "fine_tuned": ft_judge_metrics,
        },
        "custom_paired_win_rate": {
            "overall": overall_pair,
            "by_split": by_split_pair,
            "note": "Custom composite over official-style metrics; not an official EduVidQA metric.",
        },
    }

    write_json(SUMMARY_OUT, summary)
    comparison_df = summary_to_dataframe(summary)
    comparison_df.to_csv(COMPARISON_CSV_OUT, index=False)
    pair_df.to_csv(PAIRWISE_CSV_OUT, index=False)

    print(json.dumps(summary, ensure_ascii=False, indent=2)[:6000])
    print("saved summary:", SUMMARY_OUT)
    print("saved comparison csv:", COMPARISON_CSV_OUT)
    print("saved pairwise csv:", PAIRWISE_CSV_OUT)
    display(comparison_df)
    display(pair_df.head(20))

In [ ]:
# 10) Human-readable summary dashboard

def _load_current_summary():
    if "summary" in globals() and isinstance(summary, dict):
        return summary
    return read_json(SUMMARY_OUT, default={})

def _fmt_value(v):
    if v is None:
        return None
    if isinstance(v, float):
        return round(v, 4)
    return v

def _count_ok_judged(path):
    rows = read_jsonl_safe(path)
    ok = [r for r in rows if r.get("judge_status", "ok") == "ok"]
    bad = [r for r in rows if r.get("judge_status", "ok") != "ok"]
    return len(rows), len(ok), len(bad)

current_summary = _load_current_summary()
if not current_summary:
    raise RuntimeError("No summary found. Run cell #9 first, or make sure SUMMARY_OUT exists.")

base_j_total, base_j_ok, base_j_bad = _count_ok_judged(BASELINE_JUDGE_PATH)
ft_j_total, ft_j_ok, ft_j_bad = _count_ok_judged(FT_JUDGE_PATH)

completion_df = pd.DataFrame([
    {
        "model": "baseline",
        "predictions": len(baseline_preds),
        "judge_rows": base_j_total,
        "judge_ok": base_j_ok,
        "judge_failed_or_skipped": base_j_bad,
        "missing_judge_vs_predictions": max(len(baseline_preds) - base_j_ok, 0),
        "prediction_path": str(BASELINE_PRED_PATH),
        "judge_path": str(BASELINE_JUDGE_PATH),
    },
    {
        "model": "fine_tuned",
        "predictions": len(ft_preds),
        "judge_rows": ft_j_total,
        "judge_ok": ft_j_ok,
        "judge_failed_or_skipped": ft_j_bad,
        "missing_judge_vs_predictions": max(len(ft_preds) - ft_j_ok, 0),
        "prediction_path": str(FT_PRED_PATH),
        "judge_path": str(FT_JUDGE_PATH),
    },
])

rule_b = current_summary.get("rule_metrics", {}).get("baseline", {})
rule_f = current_summary.get("rule_metrics", {}).get("fine_tuned", {})
judge_b = current_summary.get("official_judge_metrics", {}).get("baseline", {})
judge_f = current_summary.get("official_judge_metrics", {}).get("fine_tuned", {})

headline_rows = []

def add_metric(group, metric, higher_is_better=True):
    b = rule_b.get(metric) if group == "rule" else judge_b.get(metric)
    f = rule_f.get(metric) if group == "rule" else judge_f.get(metric)
    if b is None and f is None:
        return
    headline_rows.append({
        "group": group,
        "metric": metric,
        "baseline": _fmt_value(b),
        "fine_tuned": _fmt_value(f),
        "delta_ft_minus_base": _fmt_value((f - b) if isinstance(b, (int, float)) and isinstance(f, (int, float)) else None),
        "higher_is_better": higher_is_better,
    })

for metric, hib in [
    ("format_pass_rate", True),
    ("no_think_rate", True),
    ("one_paragraph_rate", True),
    ("markdown_bullet_rate", False),
    ("json_or_code_fence_rate", False),
    ("empty_answer_rate", False),
    ("answer_word_count_mean", None),
    ("latency_sec_mean", False),
    ("output_tokens_mean", None),
]:
    add_metric("rule", metric, hib)

for metric in [
    "entailment_score_mean",
    "factqa_precision_mean",
    "factqa_recall_mean",
    "clarity_mean",
    "critical_thinking_mean",
    "pedagogical_techniques_mean",
]:
    add_metric("judge", metric, True)

add_metric("judge", "hallucination_proxy_1_minus_factqa_precision_mean", False)

headline_df = pd.DataFrame(headline_rows)
completion_df.to_csv(COMPLETION_STATUS_CSV, index=False)
headline_df.to_csv(HEADLINE_SUMMARY_CSV, index=False)

print("Completion status")
display(completion_df)
print("Headline metrics")
display(headline_df)

print("Saved:")
print(" -", COMPLETION_STATUS_CSV)
print(" -", HEADLINE_SUMMARY_CSV)

In [ ]:
# 11) Random qualitative review examples

RANDOM_REVIEW_N = 8
RANDOM_REVIEW_SEED = 42
RANDOM_REVIEW_MODE = "both_judged"  # "both_judged" or "any_prediction"

def _short(text, max_chars=1200):
    text = str(text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + " ...[truncated]"

def _answer_of(pred):
    return normalize_answer_text(pred)

def _score_or_none(row):
    if not row:
        return None
    try:
        return official_composite_score(row)
    except Exception:
        return None

def _verdict_from_scores(base_score, ft_score, eps=0.015):
    if base_score is None or ft_score is None:
        return "not_compared"
    delta = ft_score - base_score
    if delta > eps:
        return "FT_win"
    if delta < -eps:
        return "FT_lose"
    return "tie"

baseline_pred_map = {p["id"]: p for p in baseline_preds}
ft_pred_map = {p["id"]: p for p in ft_preds}
baseline_judge_map = {r["id"]: r for r in ok_judged(read_jsonl_safe(BASELINE_JUDGE_PATH))}
ft_judge_map = {r["id"]: r for r in ok_judged(read_jsonl_safe(FT_JUDGE_PATH))}

if RANDOM_REVIEW_MODE == "both_judged":
    candidate_ids = sorted(set(baseline_pred_map) & set(ft_pred_map) & set(baseline_judge_map) & set(ft_judge_map))
else:
    candidate_ids = sorted(set(baseline_pred_map) & set(ft_pred_map))

if not candidate_ids:
    raise RuntimeError("No candidate ids for random review. Run unified judge first or set RANDOM_REVIEW_MODE='any_prediction'.")

rng = random.Random(RANDOM_REVIEW_SEED)
sample_ids = rng.sample(candidate_ids, k=min(RANDOM_REVIEW_N, len(candidate_ids)))

review_rows = []
for id_ in sample_ids:
    ref = refs_by_id[id_]
    bp = baseline_pred_map[id_]
    fp = ft_pred_map[id_]
    bj = baseline_judge_map.get(id_)
    fj = ft_judge_map.get(id_)

    b_score = _score_or_none(bj)
    f_score = _score_or_none(fj)
    verdict = _verdict_from_scores(b_score, f_score)

    review_rows.append({
        "id": id_,
        "split": ref.get("split"),
        "question": _short(get_question_text(ref), 900),
        "reference_answer": _short(get_reference_answer(ref), 900),
        "baseline_answer": _short(_answer_of(bp), 1200),
        "fine_tuned_answer": _short(_answer_of(fp), 1200),
        "baseline_official_composite": b_score,
        "fine_tuned_official_composite": f_score,
        "verdict": verdict,
        "baseline_entailment": None if not bj else bj.get("entailment_score"),
        "fine_tuned_entailment": None if not fj else fj.get("entailment_score"),
        "baseline_factqa_precision": None if not bj else bj.get("factqa_precision"),
        "fine_tuned_factqa_precision": None if not fj else fj.get("factqa_precision"),
        "baseline_factqa_recall": None if not bj else bj.get("factqa_recall"),
        "fine_tuned_factqa_recall": None if not fj else fj.get("factqa_recall"),
        "baseline_clarity": None if not bj else bj.get("clarity"),
        "fine_tuned_clarity": None if not fj else fj.get("clarity"),
        "baseline_critical_thinking": None if not bj else bj.get("critical_thinking"),
        "fine_tuned_critical_thinking": None if not fj else fj.get("critical_thinking"),
        "baseline_pedagogical_techniques": None if not bj else bj.get("pedagogical_techniques"),
        "fine_tuned_pedagogical_techniques": None if not fj else fj.get("pedagogical_techniques"),
        "baseline_rationale": None if not bj else bj.get("judge_rationale"),
        "fine_tuned_rationale": None if not fj else fj.get("judge_rationale"),
    })

review_df = pd.DataFrame(review_rows)
write_jsonl(RANDOM_REVIEW_JSONL, review_rows)
review_df.to_csv(RANDOM_REVIEW_CSV, index=False)

md_lines = []
for r in review_rows:
    md_lines.append(f"## {r['id']} | {r['split']} | {r['verdict']}\n")
    md_lines.append(f"**Question**\n\n{r['question']}\n")
    md_lines.append(f"**Reference answer**\n\n{r['reference_answer']}\n")
    md_lines.append(f"**Baseline answer**\n\n{r['baseline_answer']}\n")
    md_lines.append(f"**Fine-tuned answer**\n\n{r['fine_tuned_answer']}\n")
    md_lines.append("**Scores**\n\n")
    md_lines.append(
        f"- composite: baseline={r['baseline_official_composite']}, fine_tuned={r['fine_tuned_official_composite']}\n"
        f"- entailment: baseline={r['baseline_entailment']}, fine_tuned={r['fine_tuned_entailment']}\n"
        f"- FactQA-P: baseline={r['baseline_factqa_precision']}, fine_tuned={r['fine_tuned_factqa_precision']}\n"
        f"- FactQA-R: baseline={r['baseline_factqa_recall']}, fine_tuned={r['fine_tuned_factqa_recall']}\n"
        f"- clarity: baseline={r['baseline_clarity']}, fine_tuned={r['fine_tuned_clarity']}\n"
        f"- critical: baseline={r['baseline_critical_thinking']}, fine_tuned={r['fine_tuned_critical_thinking']}\n"
        f"- pedagogical: baseline={r['baseline_pedagogical_techniques']}, fine_tuned={r['fine_tuned_pedagogical_techniques']}\n"
        f"- rationale baseline: {r['baseline_rationale']}\n"
        f"- rationale fine_tuned: {r['fine_tuned_rationale']}\n"
    )
    md_lines.append("\n---\n")

RANDOM_REVIEW_MD.write_text("\n".join(md_lines), encoding="utf-8")

print("Random review examples")
display(review_df)
print("Saved:")
print(" -", RANDOM_REVIEW_JSONL)
print(" -", RANDOM_REVIEW_CSV)
print(" -", RANDOM_REVIEW_MD)

In [ ]:
# 12) Token/request usage checkpoint
print("Gemini usage:")
print(json.dumps(gemini_caller.usage, ensure_ascii=False, indent=2))
print("Calls per key:", gemini_caller.calls)
print("Rate-limit/high-demand errors per key:", gemini_caller.rate_limit_errors)
print("Judge output dir:", EVAL_ONLY_DIR)